# Multiverse Analysis: Sensor and Window Robustness (Exp 2 / Passive)

Following Brockhoff et al. (2023, NeuroImage) and Steegen et al. (2016) --
repeats the main flow-effect analysis across a grid of reasonable window
widths and sensor-group sizes, reporting all of them rather than defending
one choice. This is the transparency companion to the cluster-based
permutation test: the cluster test avoids window/sensor selection entirely,
this instead exhausts the reasonable alternatives and shows whether the
conclusion holds across all of them.

For each component x window-width x sensor-group combination: a paired
t-test on mean amplitude (High vs Low flow). Where p >= .05, a Bayesian
paired t-test (JZS prior, r=0.707, matching Brockhoff's JASP convention)
quantifies evidence for/against the null, via `pingouin` -- same default
prior JASP uses.

Last edit: 19.10.2026 @anguyen

## Setup: configuration, paths, and cached difference-wave evokeds

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import time

def find_repo_root(marker='core') -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(f"Could not locate repo root (no '{marker}/' found above {Path.cwd()})")

sys.path.insert(0, str(find_repo_root() / 'core'))
from config_flow import CONDITIONS
from participants import get_participants
from utils import load_pkl, show_latest_pkls

PAPER:    str = 'flow'
PARADIGM: str = 'passive'

STUDY = 'Active' if PARADIGM == 'active' else 'Passive'
if sys.platform == 'win32':
    root = Path(f'L:/EEG_Flow_data/Study_EEG_Flow_{STUDY}/derivatives_v2')
else:
    root = Path(f'/Volumes/faculty/EEG_Flow_data/Study_EEG_Flow_{STUDY}/derivatives_v2')

DATE_SOURCE = '20261015'

n = len(get_participants(PAPER, PARADIGM))

DEVIANT_LABEL   = 'target' if PARADIGM == 'active' else 'deviant'
COND_HIGH_LABEL = 'High Flow'
COND_LOW_LABEL  = 'Low Flow'
cond_key_high   = CONDITIONS[COND_HIGH_LABEL]
cond_key_low    = CONDITIONS[COND_LOW_LABEL]

OUTPUT_DIR = root / 'outputs' / 'stats' / 'flow' / 'multiverse'
PLOTS_DIR  = root / 'outputs' / 'img'   / 'flow' / 'multiverse'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True,  exist_ok=True)

print(f'Paper: {PAPER.upper()} | Paradigm: {PARADIGM.upper()} | N={n}')
print(f'DATE_SOURCE: {DATE_SOURCE}')
print(f'Condition labels: {list(CONDITIONS.keys())}')

In [ ]:
input_path_diff = root / 'outputs' / 'grand_averages' / PAPER / 'diff'

def _build_fname_diff(cond_key: str) -> str:
    return f'{DATE_SOURCE}_{PARADIGM}_{n}p_cond-{cond_key}_stim_diff.pkl'

evokeds_diff: dict[str, dict] = {}
for plot_label, cond_key in CONDITIONS.items():
    evokeds_diff[plot_label] = load_pkl(input_path_diff / _build_fname_diff(cond_key))
    print(f'Loaded [{plot_label}] -- diff keys: {list(evokeds_diff[plot_label].keys())}')

## Specification grid

Each contrast has multiple established components, each with its own
window -- not one window per contrast. Using your actual values:

- **Deviant-Standard**: N1 (90-120ms), MMN (120-180ms), P3a (200-280ms)
- **Novel-Standard**: N1+MMN grouped (120-180ms), P3a (180-280ms)

**Two different width-variation modes**, matching Brockhoff's actual
approach per component type (not one symmetric rule for everything):
- **Transient components (N1, MMN, N1+MMN)**: symmetric total-width change
  around the same center -- e.g. Brockhoff's N1: 50ms interval becomes
  30ms/70ms, +/-20ms **total**, not per side.
- **P3a (broader, fuzzy offset)**: only the END boundary varies, start
  stays fixed -- e.g. Brockhoff's P3: 300-700ms becomes 300-600ms/300-800ms,
  +/-100ms on the offset only. Onset is well-defined; offset isn't, so only
  that boundary is stress-tested.

Sensor sets still expand/contract in **both** anterior-posterior and
lateral directions together (not narrower in one dimension only), absent a
specific topographic reason to favor one axis.

In [ ]:
def symmetric_width_variants(main_window, total_delta_ms):
    center = (main_window[0] + main_window[1]) / 2
    half_width = (main_window[1] - main_window[0]) / 2
    half_delta = total_delta_ms / 2
    if half_delta >= half_width:
        raise ValueError(
            f'delta ({total_delta_ms}ms) too large for window {main_window} '
            f'(width {main_window[1]-main_window[0]}ms).'
        )
    return {
        'narrower': (center - (half_width - half_delta), center + (half_width - half_delta)),
        'main':     main_window,
        'wider':    (center - (half_width + half_delta), center + (half_width + half_delta)),
    }


def end_only_variants(main_window, delta_ms):
    start, end = main_window
    width = end - start
    if delta_ms >= width:
        raise ValueError(
            f'delta ({delta_ms}ms) too large for window {main_window} '
            f'(width {width}ms). Try roughly 25% of the window width.'
        )
    return {
        'narrower': (start, end - delta_ms),
        'main':     main_window,
        'wider':    (start, end + delta_ms),
    }


DEV_COMPONENT_WINDOWS = {
    'N1':  {'window': (90,  120), 'delta': 20, 'mode': 'symmetric'},
    'MMN': {'window': (120, 180), 'delta': 20, 'mode': 'symmetric'},
    'P3a': {'window': (200, 280), 'delta': 20, 'mode': 'end_only'},
}
NOV_COMPONENT_WINDOWS = {
    'N1_MMN': {'window': (120, 180), 'delta': 20, 'mode': 'symmetric'},
    'P3a':    {'window': (180, 280), 'delta': 25, 'mode': 'end_only'},
}

def build_windows(component_windows: dict) -> dict:
    out = {}
    for comp, spec in component_windows.items():
        fn = symmetric_width_variants if spec['mode'] == 'symmetric' else end_only_variants
        out[comp] = fn(spec['window'], spec['delta'])
    return out

DEV_WINDOWS_BY_COMPONENT = build_windows(DEV_COMPONENT_WINDOWS)
NOV_WINDOWS_BY_COMPONENT = build_windows(NOV_COMPONENT_WINDOWS)

FRONTAL         = ['F1',  'Fz',  'F2',  'FC1', 'FCz', 'FC2']
CENTRAL         = ['FC1', 'FCz', 'FC2', 'C1',  'Cz',  'C2']
CENTRO_PARIETAL = ['C1',  'Cz',  'C2',  'CP1', 'CPz', 'CP2']

DEV_SENSOR_SETS_BY_COMPONENT = {
    'N1': {
        'fewer': ['FC1', 'FCz', 'FC2'],
        'main':  FRONTAL,
        'more':  FRONTAL + ['F3', 'F4', 'FC3', 'FC4'],
    },
    'MMN': {
        'fewer': ['FC1', 'FCz', 'FC2'],
        'main':  FRONTAL,
        'more':  FRONTAL + ['F3', 'F4', 'FC3', 'FC4'],
    },
    'P3a': {
        'fewer': ['C1', 'Cz', 'C2'],
        'main':  CENTRAL,
        'more':  CENTRAL + ['FC3', 'FC4', 'C3', 'C4'],
    },
}
NOV_SENSOR_SETS_BY_COMPONENT = {
    'N1_MMN': {
        'fewer': ['CP1', 'CPz', 'CP2'],
        'main':  CENTRO_PARIETAL,
        'more':  CENTRO_PARIETAL + ['C3', 'C4', 'CP3', 'CP4'],
    },
    'P3a': {
        'fewer': ['C1', 'Cz', 'C2'],
        'main':  CENTRAL,
        'more':  CENTRAL + ['FC3', 'FC4', 'C3', 'C4'],
    },
}

for comp, variants in DEV_WINDOWS_BY_COMPONENT.items():
    print(f'Deviant-Standard [{comp}]: {variants}')
for comp, variants in NOV_WINDOWS_BY_COMPONENT.items():
    print(f'Novel-Standard [{comp}]: {variants}')

## Per-subject mean amplitude extraction

In [ ]:
def per_subject_mean_amplitude(diff_event: str, sensors: list[str], window_ms: tuple):
    evokeds_high_raw = evokeds_diff[COND_HIGH_LABEL][f'{cond_key_high}_{diff_event}']
    evokeds_low_raw  = evokeds_diff[COND_LOW_LABEL][f'{cond_key_low}_{diff_event}']

    high_by_subj = {ev.comment: ev for ev in evokeds_high_raw}
    low_by_subj  = {ev.comment: ev for ev in evokeds_low_raw}
    common_subjects = sorted(set(high_by_subj) & set(low_by_subj))

    high_vals, low_vals = [], []
    for s in common_subjects:
        h = high_by_subj[s].copy().pick(sensors)
        l = low_by_subj[s].copy().pick(sensors)
        times_ms = h.times * 1000
        mask = (times_ms >= window_ms[0]) & (times_ms <= window_ms[1])
        high_vals.append(h.data[:, mask].mean() * 1e6)
        low_vals.append(l.data[:, mask].mean() * 1e6)

    return common_subjects, np.array(high_vals), np.array(low_vals)

## Run the full 3x3 grid per component

Paired t-test for every specification; Bayesian paired t-test (BF10/BF01)
added specifically where p >= .05, following Brockhoff/Dienes (2016)/
Keysers et al. (2020). Requires `pingouin` (`pip install pingouin
--break-system-packages` if not already installed).

In [ ]:
import pingouin as pg

def _get_col(result: pd.DataFrame, *possible_names):
    for name in possible_names:
        if name in result.columns:
            return result[name].iloc[0]
    raise KeyError(f'None of {possible_names} found in pingouin output: {result.columns.tolist()}')


def run_multiverse(
    diff_event: str,
    windows_by_component: dict,
    sensor_sets_by_component: dict,
    contrast_label: str,
) -> pd.DataFrame:
    t0   = time.time()
    rows = []
    for component, windows in windows_by_component.items():
        sensor_sets = sensor_sets_by_component[component]
        for width_label, window_ms in windows.items():
            for sensor_label, sensors in sensor_sets.items():
                subjects, high_vals, low_vals = per_subject_mean_amplitude(
                    diff_event, sensors, window_ms
                )
                result = pg.ttest(high_vals, low_vals, paired=True)
                p_val  = _get_col(result, 'p_val', 'p-val')
                t_val  = _get_col(result, 'T', 't')
                bf10   = float(_get_col(result, 'BF10', 'bf10'))

                row = {
                    'contrast':    contrast_label,
                    'component':   component,
                    'width':       width_label,
                    'sensors':     sensor_label,
                    'n_sensors':   len(sensors),
                    'window_ms':   f'{window_ms[0]:.0f}-{window_ms[1]:.0f}',
                    'N':           len(subjects),
                    't':           round(t_val, 3),
                    'p':           round(p_val, 4),
                    'significant': p_val < 0.05,
                }
                if p_val >= 0.05:
                    row['BF10']    = round(bf10, 2)
                    row['BF01']    = round(1 / bf10, 2)
                    row['bf_note'] = 'reported per Dienes 2016 / Keysers et al. 2020 (p >= .05)'
                else:
                    row['BF10']    = None
                    row['BF01']    = None
                    row['bf_note'] = ''
                rows.append(row)

    df    = pd.DataFrame(rows)
    n_sig = df['significant'].sum()
    print(f'[{contrast_label}] {n_sig} / {len(df)} specifications significant (p < .05)')
    elapsed = time.time() - t0
    print(f'Done in {int(elapsed//60)}m {elapsed%60:.1f}s')
    return df


dev_results = run_multiverse(
    f'{DEVIANT_LABEL}-standard',
    DEV_WINDOWS_BY_COMPONENT,
    DEV_SENSOR_SETS_BY_COMPONENT,
    'Deviant-Standard',
)
dev_results

In [ ]:
novel_results = run_multiverse(
    'novel-standard',
    NOV_WINDOWS_BY_COMPONENT,
    NOV_SENSOR_SETS_BY_COMPONENT,
    'Novel-Standard',
)
novel_results

## Save results

In [ ]:
all_results = pd.concat([dev_results, novel_results], ignore_index=True)

CLUSTER_NAME_BY_COMPONENT = {
    ('Deviant-Standard', 'N1'):    'Frontal',
    ('Deviant-Standard', 'MMN'):   'Frontal',
    ('Deviant-Standard', 'P3a'):   'Central',
    ('Novel-Standard',   'N1_MMN'):'Centro-parietal',
    ('Novel-Standard',   'P3a'):   'Central',
}

all_results['Window']  = all_results.apply(lambda r: f"{r['width']} ({r['window_ms']})", axis=1)
all_results['Cluster'] = all_results.apply(
    lambda r: CLUSTER_NAME_BY_COMPONENT[(r['contrast'], r['component'])], axis=1
)
all_results['Size'] = all_results['sensors']
all_results = all_results.drop(columns=['width', 'window_ms', 'sensors', 'n_sensors', 'bf_note'])
all_results = all_results[
    ['contrast', 'component', 'Window', 'Cluster', 'Size', 'N', 't', 'p', 'significant', 'BF10', 'BF01']
]

# Electrode reference table
electrode_ref_rows = []
for cluster_name, sensor_dict in [
    ('Frontal',          DEV_SENSOR_SETS_BY_COMPONENT['MMN']),
    ('Central',          DEV_SENSOR_SETS_BY_COMPONENT['P3a']),
    ('Centro-parietal',  NOV_SENSOR_SETS_BY_COMPONENT['N1_MMN']),
]:
    for size_label, electrodes in sensor_dict.items():
        electrode_ref_rows.append({
            'Cluster':      cluster_name,
            'Size':         size_label,
            'N electrodes': len(electrodes),
            'Electrodes':   ', '.join(electrodes),
        })
electrode_reference = pd.DataFrame(electrode_ref_rows)

out_path = OUTPUT_DIR / 'multiverse_flow_effect_exp2.csv'
all_results.to_csv(out_path, index=False)
electrode_reference.to_csv(OUTPUT_DIR / 'multiverse_electrode_reference.csv', index=False)
print(f'Saved: {out_path}')
print(f'Saved: {OUTPUT_DIR / "multiverse_electrode_reference.csv"}')
all_results

## Specification curve plot

Sorted effect sizes/p-values across all 9 specifications per component --
the standard visualization for a multiverse/specification-curve analysis
(Simonsohn et al., 2020).

In [ ]:
import matplotlib.pyplot as plt

def plot_specification_curve(df: pd.DataFrame, contrast_label: str):
    components = df['component'].unique()
    fig, axes  = plt.subplots(1, len(components), figsize=(5 * len(components), 4), squeeze=False)

    for ax, component in zip(axes[0], components):
        sub    = df[df['component'] == component].sort_values('t').reset_index(drop=True)
        colors = ['crimson' if sig else 'steelblue' for sig in sub['significant']]
        ax.scatter(range(len(sub)), sub['t'], c=colors, s=60, zorder=3)
        ax.axhline(0, color='grey', linewidth=0.8)
        ax.set_xticks(range(len(sub)))
        ax.set_xticklabels(
            [f"{r.width}/{r.sensors}" for r in sub.itertuples()],
            rotation=45, ha='right', fontsize=7
        )
        ax.set_ylabel('t-statistic (High vs Low)')
        ax.set_title(component)

    fig.suptitle(f'Specification curve: {contrast_label} (red = p<.05, blue = p>=.05)')
    fig.tight_layout()

    fname = PLOTS_DIR / f'specification_curve_{contrast_label.lower().replace(" ", "_").replace("-", "_")}.png'
    fig.savefig(fname, dpi=150, bbox_inches='tight')
    print(f'Saved: {fname}')
    plt.show()


plot_specification_curve(dev_results,   'Deviant-Standard')
plot_specification_curve(novel_results, 'Novel-Standard')